```mermaid
flowchart LR
    A0["00"] --> A1a["01a"] --> A1b["01b"] --> A2["02"] --> A3["03"] --> A4a["04a"] --> A4b["04b"]
    A4b --> A5a["05a"] --> A5b["05b"] --> A6a["06a"] --> A6b["06b"]
    A6b --> A7["07"] --> A8a["08a"] --> A8b["08b"]
    A8b --> A9["09"] --> A10["10"] --> A11["11"] --> A12["12"] 
    
    classDef normal fill:#f8f9fa,stroke:#adb5bd,stroke-width:1px,color:#111;
    classDef done fill:#e8f7f0,stroke:#198754,stroke-width:1.5px,color:#111;
    classDef current fill:#fff3cd,stroke:#ff8c00,stroke-width:2px,color:#111;
    
    class A0,A1a,A1b,A2,A3,A4a,A4b,A5a,A5b,A6a,A6b,A7,A8a done;
    class A8b current;
    class A9,A10,A11,A12 normal;
```

# Notebook 08b — What Transformers Are: The Model Behind Our Embeddings

This notebook is a **conceptual companion** to Notebook 08. In Notebook 08, we generated and used embeddings with the model `sentence-transformers/all-MiniLM-L6-v2`. Here, we pause to ask a methodological question:

> What kind of model produced those embeddings, and why does it work well for similarity-based NLP tasks?

This notebook is deliberately **light on code** and **heavy on interpretation**. The goal is not to train a transformer model, and not to derive the mathematics of attention. Instead, the aim is to build a clear conceptual understanding of the model family you have already used in practice.

We will focus on:
- what transformers are, in broad terms
- what makes embeddings from transformer-based models different from Bag-of-Words or TF–IDF features
- the intuition behind **attention**
- why `MiniLM` is small enough for a laptop-friendly course workflow
- why sentence-transformer models are useful for semantic search and similarity
- what these models can and cannot justify interpretively


## Learning goals

By the end of this notebook, students should be able to:

- explain, in non-technical terms, what a transformer model is
- distinguish **classical lexical features** from **contextual embeddings**
- describe the intuition behind **attention**
- explain why `all-MiniLM-L6-v2` is a practical model choice for this course
- understand what `sentence-transformers` adds on top of a base transformer
- critically interpret embedding-based outputs without treating them as direct evidence of meaning


## Why this notebook appears here

The placement of this notebook in the course is intentional. By this point, you have already used a transformer-based embedding model in Notebook 08. You have seen that embeddings can support tasks such as:

- semantic search
- nearest-neighbor retrieval
- similarity comparison
- geometric visualization of texts

Rather than introducing transformers first in the abstract, we now explain them **after you have used them**. This allows the conceptual explanation to attach to concrete notebook experience rather than to an unfamiliar technical architecture.

This is also methodologically important: modern NLP tools often look persuasive because their outputs are smooth, numeric, and apparently semantic. A course like this should therefore not only teach how to run such models, but also what kinds of representations they produce and what kinds of interpretive caution they require.


## From Bag-of-Words to dense embeddings

In earlier notebooks, we worked with **Bag-of-Words** and **TF–IDF** representations. These are extremely useful because they are transparent, interpretable, and efficient. But they represent texts mainly through explicit lexical content: which words appear, how often they appear, and how distinctive they are relative to the corpus.

Transformer-based embeddings work differently. Instead of representing a text as a very large sparse vector indexed by words or n-grams, they produce a **dense vector**: a relatively compact numerical representation that aims to capture patterns of contextual usage learned from large corpora.

The contrast can be summarized like this:

| Representation | Typical shape | Main intuition | Strength | Limitation |
|---|---|---|---|---|
| Bag-of-Words | sparse | count words | interpretable | ignores semantics |
| TF–IDF | sparse | weight distinctive words | strong lexical baseline | still mostly lexical overlap |
| Transformer embedding | dense | encode contextual usage patterns | captures broader similarity | less directly interpretable |

A useful simplification is this: classical features ask **what words are here?** Transformer embeddings ask something closer to **what kind of context or usage pattern does this text resemble?**


## What is a transformer?

A **transformer** is a neural network architecture designed to model language by representing each token in relation to the other tokens around it. The key conceptual shift is that words are not processed as isolated items with fixed meanings. Instead, the model builds **context-sensitive representations**.

That means the representation of a word depends on the surrounding text. For example, a word such as `reason`, `nature`, or `law` may appear in many philosophical contexts. A transformer does not assume that each occurrence has one stable vector regardless of context. It instead computes a representation shaped by nearby words and broader sentence structure.

This is one reason transformer-based models are often more effective than earlier embedding approaches for semantic tasks. They can represent the same word differently depending on whether it appears in, for example:

- a metaphysical discussion
- a moral argument
- a discussion of natural science
- a critique of another philosopher

In short: transformers model language as **context-dependent**, not as a simple lookup table of fixed word meanings.


## Key term: contextual embeddings

A **contextual embedding** is a vector representation whose value depends on context. This is different from older embedding models such as Word2Vec or GloVe, where a word typically has one main vector regardless of where it appears.

Why does this matter for the course? Because many of the concepts in philosophical texts are **historically variable, abstract, and polysemous**. A fixed lexical representation often misses these contextual differences. Contextual embeddings do not solve the interpretive problem completely, but they provide a richer representation for tasks such as semantic search, retrieval, and neighborhood comparison.

Still, it is important to be precise: contextual embeddings are **not direct models of philosophical meaning**. They are numerical summaries of usage patterns learned from data. They can be helpful, but they remain model-dependent approximations.


## Attention: the core intuition

The most famous idea associated with transformers is **attention**. In a full technical treatment, attention involves query, key, and value matrices. In this course, we do not need the equations. What matters is the underlying intuition.

Attention is a mechanism that allows the model to ask something like:

> Which other words in this sequence are most relevant for interpreting the current word?

When the model processes a token, it can assign more or less weight to other tokens in the same input. That is, it does not treat all neighboring words as equally relevant. Instead, it learns patterns about which parts of the context matter most for representing a token or for solving a training objective.

This helps explain why transformers are effective at capturing relational meaning. A token can be represented in light of:

- nearby modifiers
- long-distance dependencies
- recurring lexical associations
- broader sentence-level structure

So a useful non-technical definition is: **attention is a learned way of weighting context**.


## Why attention matters for interpretation

Attention is sometimes described in very strong terms, as if it reveals exactly what the model 'understands' or 'cares about'. That is too strong. Attention is best treated as part of the model's internal mechanism for building useful representations, not as a transparent explanation of human-like understanding.

For this course, the important methodological lesson is more modest: attention helps explain why transformer models can produce embeddings that are often more semantically useful than purely lexical features. But attention does **not** mean that the model has human interpretive insight, and it does **not** remove the need for critical reading of outputs.


## From token representations to sentence embeddings

A base transformer usually produces **token-level contextual representations**. But in Notebook 08, we did not mainly need token-level vectors. We needed one vector per **chunk** or per short text span, so that we could compare chunks by semantic similarity.

This is where `sentence-transformers` becomes important. Sentence-transformer models are designed to convert whole sentences or short passages into vectors that work well for tasks such as:

- semantic search
- retrieval
- clustering
- similarity ranking

Conceptually, this means two things happened in Notebook 08:

1. a transformer produced contextual internal representations
2. the sentence-transformer framework pooled and optimized those representations for sentence- or chunk-level similarity

So the model you used was not just 'a transformer' in the abstract. It was a transformer adapted for **sentence embedding** tasks.


## Why `all-MiniLM-L6-v2` is small

The model used in the course, `sentence-transformers/all-MiniLM-L6-v2`, is deliberately much smaller than many large transformer models. This matters in a course that is designed for **CPU-friendly, student-laptop workflows**.

A few intuitive points are enough here:

- `MiniLM` is a **compressed** or **distilled** transformer-style model
- it has fewer layers and parameters than large language models used in more demanding settings
- it is therefore faster to load, faster to run, and lighter on memory
- it still performs well enough for many embedding tasks such as similarity and retrieval

In other words, MiniLM is a practical compromise: it gives access to transformer-based semantic representations without requiring the computational resources needed for large-scale training or very large inference pipelines.

For this course, that trade-off is pedagogically appropriate. The goal is not to maximize benchmark performance at any cost, but to give students a usable modern NLP representation that can be inspected critically and run reproducibly on modest hardware.


## What transformers add — and what they do not add

Transformer-based embeddings add something important to the course toolkit: they often capture broader similarity than lexical methods alone. This means they can retrieve semantically related chunks even when those chunks do not share many exact words.

But it is equally important to state what they do **not** automatically add:

- they do not guarantee philosophical understanding
- they do not solve historical ambiguity
- they do not eliminate corpus bias
- they do not remove the need for representative text inspection
- they do not turn similarity scores into proof of semantic equivalence

That is why the course repeatedly emphasizes **evaluation-by-inspection**. A retrieved neighbor, a cluster, or a semantic shift pattern should always be interpreted in relation to actual textual evidence and corpus composition, not accepted automatically because the vectors look convincing.


## Lightweight inspection: what model are we using?

The following code cell is optional. It simply loads the same sentence-transformer model used earlier and prints a small amount of metadata. The point is not technical analysis, but to make the model object itself a little less abstract.


In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    HAS_ST = True
except Exception:
    HAS_ST = False
    print('sentence-transformers not available.')

EMBEDDING_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'

if HAS_ST:
    print('Loading model:', EMBEDDING_MODEL_NAME)
    model = SentenceTransformer(EMBEDDING_MODEL_NAME)
    print('Model loaded.')
    print(model)

## Similarity is broader than exact word overlap

This tiny demo is not meant as proof of semantic understanding. It is simply a reminder of why embedding models are attractive in practice: they can place short texts near each other even when they do not share exactly the same wording.

If the library is available, the next cell encodes a few short phrases and compares them with cosine similarity. The result should be read as an informal demonstration of representational behavior, not as a formal evaluation.


In [ ]:
import numpy as np

try:
    from sklearn.metrics.pairwise import cosine_similarity
except Exception:
    cosine_similarity = None

examples = [
    'virtue and moral character',
    'ethical excellence and character',
    'astronomical measurement and stars',
    'reason and rational judgment',
]

if HAS_ST and cosine_similarity is not None:
    V = model.encode(examples, normalize_embeddings=True, convert_to_numpy=True)
    S = cosine_similarity(V)
    print('Example phrases:')
    for i, text in enumerate(examples):
        print(f'{i}: {text}')
    print('\nCosine similarity matrix:')
    print(np.round(S, 3))
else:
    print('Optional demo skipped because required packages are unavailable.')


## Transformer NER as a comparison point

If you want, you can also treat transformers as a comparison point for Named Entity Recognition. A transformer-based NER model can sometimes detect entities differently from spaCy's small pretrained pipeline. In a course like this, such a comparison can be useful for discussing **domain mismatch**, **model size**, and the difference between **inference quality** and **computational cost**.

However, this should remain an optional demonstration rather than a new required pipeline. The main conceptual point is that transformer architectures are not only used for embeddings; they also underpin many modern sequence-labeling, classification, and generation systems.


## Reflection questions

1. In what sense are transformer embeddings more informative than TF–IDF, and in what sense are they less interpretable?
2. Why is it useful to say that transformer embeddings are **contextual** rather than simply 'semantic'?
3. What does the idea of attention add to your intuition about language modeling, even without the equations?
4. Why is `MiniLM` a particularly suitable choice for a CPU-friendly, methods-first course?
5. Why should embedding similarity still be checked against retrieved passages rather than treated as direct proof of conceptual similarity?


## Method note

This notebook has intentionally avoided the full mathematical formulation of transformer models. That is not because the technical details are unimportant, but because the course goal here is methodological literacy rather than deep-learning specialization. Students should leave with a conceptually sound understanding of what these models are doing and why their outputs can be useful, while still recognizing that modern NLP systems remain probabilistic, trained artifacts rather than neutral instruments of interpretation.


## Conclusion

In Notebook 08, you used embeddings as practical tools for search, similarity, and comparison. In this notebook, we stepped back and clarified the model family behind those embeddings. The key takeaway is not that transformers are magical or opaque black boxes that must simply be trusted. Rather, the key takeaway is that they provide a different representational logic from classical lexical methods: they model words and texts in relation to context, and they can therefore support richer similarity behavior than Bag-of-Words or TF–IDF alone.

At the same time, richer representation does not eliminate interpretive responsibility. In the rest of the course, transformer-based outputs should be treated as analytically useful but methodologically bounded. They can help reveal patterns, retrieve related passages, and support exploratory claims. But they still require critical inspection, comparison with alternative representations, and careful attention to corpus structure, bias, and uncertainty.

In that sense, this notebook completes the pedagogical loop: you have not only used a transformer-derived model, but also learned what kind of model it is and how to interpret its outputs responsibly.


```mermaid
flowchart TB
    A0["00<br/>Bootcamp"] --> P1

    subgraph P1["Part I — Corpus building and analysis"]
        direction LR
        A1a["01a<br/>Corpus metadata"] --> A1b["01b<br/>Corpus building"] --> A2["02<br/>Preprocessing"] --> A3["03<br/>Distributions + time"] --> A4a["04a<br/>Lexical exploration"] --> A4b["04b<br/>Embedding"]
    end

    subgraph P2["Part II — Linguistic annotations"]
        direction LR
        A5a["05a<br/>spaCy annotation"] --> A5b["05b<br/>Relation extraction"] --> A6a["06a<br/>NER"] --> A6b["06b<br/>Custom NER"]
    end

    subgraph P3["Part III — Representations"]
        direction LR
        A7["07<br/>BoW + TF-IDF"] --> A8a["08a<br/>Embeddings"] --> A8b["08b<br/>Transformers"]
    end

    subgraph P4["Part IV — Models and interpretation"]
        direction LR
        A9["09<br/>Classification"] --> A10["10<br/>Custom NER training"] --> A11["11<br/>Topic modeling"] --> A12["12<br/>Semantic shift"]
    end

    P1 --> P2
    P2 --> P3
    P3 --> P4

    classDef start fill:#f3f0ff,stroke:#6f42c1,stroke-width:1.5px,color:#111;
    classDef prep fill:#eef7ff,stroke:#1f77b4,stroke-width:1.5px,color:#111;
    classDef annot fill:#eefaf0,stroke:#2ca02c,stroke-width:1.5px,color:#111;
    classDef repr fill:#fff7e6,stroke:#ff8c00,stroke-width:1.5px,color:#111;
    classDef model fill:#fff0f0,stroke:#d62728,stroke-width:1.5px,color:#111;

    classDef highlight fill:#fff3b0,stroke:#f5a623,stroke-width:4px,color:#111;

    class A1a,A1b,A2,A3,A4a,A4b prep;
    class A5a,A5b,A6a,A6b annot;
    class A7,A8a,A8b repr;
    class A9,A10,A11,A12 model;

    class A8b highlight;
```